In [35]:
!pip install -q pymupdf sentence-transformers faiss-cpu transformers==4.57.1 sentencepiece

In [48]:
from google.colab import files
import fitz

uploaded = files.upload()
pdf_path = next(iter(uploaded))

doc = fitz.open(pdf_path)

text = ""
for page in doc:
    text += page.get_text() + "\n"

print("PDF extracted successfully!")
print(text[:1000])

Saving 2410.15944v1.pdf to 2410.15944v1 (1).pdf
PDF extracted successfully!
Developing Retrieval Augmented Generation
(RAG) based LLM Systems from PDFs: An
Experience Report
Ayman Asad Khan
Tampere University
ayman.khan@tuni.fi
Md Toufique Hasan
Tampere University
mdtoufique.hasan@tuni.fi
Kai Kristian Kemell
Tampere University
kai-kristian.kemell@tuni.fi
Jussi Rasku
Tampere University
jussi.rasku@tuni.fi
Pekka Abrahamsson
Tampere University
pekka.abrahamsson@tuni.fi
Abstract. This paper presents an experience report on the develop-
ment of Retrieval Augmented Generation (RAG) systems using PDF
documents as the primary data source. The RAG architecture combines
generative capabilities of Large Language Models (LLMs) with the preci-
sion of information retrieval. This approach has the potential to redefine
how we interact with and augment both structured and unstructured
knowledge in generative models to enhance transparency, accuracy and
contextuality of responses. The paper details the

In [51]:
def split_text(text, chunk_size=200, overlap=40):

    words = text.split()
    chunks = []

    step = chunk_size - overlap

    for i in range(0, len(words), step):

        chunk = " ".join(words[i:i + chunk_size])

        if len(chunk.split()) > 30:
            chunks.append(chunk)

    return chunks


chunks = split_text(text)

print("Total words:", len(text.split()))
print("Total chunks:", len(chunks))



Total words: 9787
Total chunks: 61


In [50]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True,
    normalize_embeddings=True
)

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings.astype("float32"))

print("FAISS index created successfully!")
print("Embedding shape:", embeddings.shape)

FAISS index created successfully!
Embedding shape: (61, 384)


In [52]:
!pip install -q rank-bm25

In [53]:
from rank_bm25 import BM25Okapi

# Create BM25 keyword index
tokenized_chunks = [
    chunk.lower().split() for chunk in chunks
]

bm25 = BM25Okapi(tokenized_chunks)

print("BM25 index created successfully!")

BM25 index created successfully!


In [54]:
def improved_retrieve(question, k=5):

    q = question.lower()

    if "what is" in q and "rag" in q:
        extra = """
        definition combines generative capabilities
        large language models precision information retrieval
        """

    elif "main stages" in q or "pipeline" in q:
        extra = """
        key steps data collection document processing
        chunking embeddings vector database retrieval generation
        """

    elif "limitations" in q or "drawbacks" in q:
        extra = """
        challenges drawbacks complex PDF layouts
        multi-column embedded images retrieval efficiency
        irregular structures text extraction chunking
        """

    elif "improvements" in q or "improve" in q:
        extra = """
        improvements feedback better text extraction
        chunking document processing retrieval efficiency
        """

    else:
        extra = ""

    expanded_query = question + " " + extra

    # Semantic search using FAISS
    query_embedding = embedding_model.encode(
        [expanded_query],
        normalize_embeddings=True
    )

    scores, dense_indices = index.search(
        query_embedding.astype("float32"), 8
    )

    # Keyword search using BM25
    keyword_scores = bm25.get_scores(
        expanded_query.lower().split()
    )

    keyword_indices = np.argsort(keyword_scores)[::-1][:8]

    # Reciprocal Rank Fusion
    rrf_scores = {}

    for rank, idx in enumerate(dense_indices[0]):

        if idx >= 0:
            rrf_scores[idx] = (
                rrf_scores.get(idx, 0) + 1 / (60 + rank + 1)
            )

    for rank, idx in enumerate(keyword_indices):

        rrf_scores[idx] = (
            rrf_scores.get(idx, 0) + 1 / (60 + rank + 1)
        )

    ranked_indices = sorted(
        rrf_scores,
        key=rrf_scores.get,
        reverse=True
    )

    return [chunks[i] for i in ranked_indices[:k]]


print("Hybrid retrieval function created!")

Hybrid retrieval function created!


In [55]:
from transformers import pipeline

generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-small"
)

def ask_question(question):

    results = improved_retrieve(question, k=3)
    context = "\n".join(results)

    prompt = f"""
    Answer the question using the context.

    Rules:
    1. Give a direct answer in 2 sentences.
    2. Use simple English.
    3. Do not include section numbers or headings.
    4. Avoid copying unrelated text.
    5. Do not add information not present in context.

    Context:
    {context[:1500]}

    Question: {question}

    Answer:
    """

    result = generator(
        prompt,
        max_new_tokens=100,
        truncation=True
    )

    return result[0]["generated_text"].strip()

Device set to use cpu


In [56]:
questions = [
    "What is Retrieval-Augmented Generation?",
    "Why are PDF documents used in RAG?",
    "What are the main stages of a RAG pipeline?",
    "What are the limitations of RAG systems?",
    "What improvements can be made to RAG systems?"
]

for q in questions:
    print("\nQuestion:", q)
    print("Answer:", ask_question(q))
    print("-" * 60)


Question: What is Retrieval-Augmented Generation?
Answer: RAG models are built on in- tegrating two core components of NLP: Information Retrieval (IR) and Natural Language Generation (NLG). The RAG framework, first introduced by Lewis et al.[5] combines dense retrieval methods with large scale generative models to produce responses that are both contextually relevant and factually accurate. By explicitly retrieving relevant passages from a large corpus and augmenting this information in the
------------------------------------------------------------

Question: Why are PDF documents used in RAG?
Answer: they are widely used for distributing high-value content like research papers, legal documents, technical manuals, and financial reports, all of which contain dense, detailed information essential for training RAG models
------------------------------------------------------------

Question: What are the main stages of a RAG pipeline?
Answer: a retriever, which converts user queries in

In [57]:
from transformers import pipeline

generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-small"
)

print("FLAN-T5 model loaded successfully!")

Device set to use cpu


FLAN-T5 model loaded successfully!


In [58]:
def ask_question(question):

    results = improved_retrieve(question, k=3)

    context = "\n".join(results)

    prompt = f"""
    Answer the question using the context.

    Rules:
    1. Give a direct answer in 2 sentences.
    2. Use simple English.
    3. Do not include section numbers or headings.
    4. Avoid copying unrelated text.
    5. Do not add information not present in context.

    Context:
    {context[:1500]}

    Question: {question}

    Answer:
    """

    result = generator(
        prompt,
        max_new_tokens=100,
        truncation=True
    )

    return result[0]["generated_text"].strip()


print("RAG question-answering system is ready!")

RAG question-answering system is ready!


In [59]:
questions = [
    "What is Retrieval-Augmented Generation?",
    "Why are PDF documents used in RAG?",
    "What are the main stages of a RAG pipeline?",
    "What are the limitations of RAG systems?",
    "What improvements can be made to RAG systems?"
]

for q in questions:

    print("\nQuestion:", q)
    print("Answer:", ask_question(q))
    print("-" * 60)


Question: What is Retrieval-Augmented Generation?
Answer: RAG models are built on in- tegrating two core components of NLP: Information Retrieval (IR) and Natural Language Generation (NLG). The RAG framework, first introduced by Lewis et al.[5] combines dense retrieval methods with large scale generative models to produce responses that are both contextually relevant and factually accurate. By explicitly retrieving relevant passages from a large corpus and augmenting this information in the
------------------------------------------------------------

Question: Why are PDF documents used in RAG?
Answer: they are widely used for distributing high-value content like research papers, legal documents, technical manuals, and financial reports, all of which contain dense, detailed information essential for training RAG models
------------------------------------------------------------

Question: What are the main stages of a RAG pipeline?
Answer: a retriever, which converts user queries in

In [60]:
while True:

    question = input("\nAsk a question (type exit to stop): ")

    if question.lower() == "exit":
        print("Thank you!")
        break

    answer = ask_question(question)

    print("\nAnswer:", answer)


Ask a question (type exit to stop): what is fiass

Answer: a re- trieved context with OLlama and Llama 3.1. Common Mistakes and Best Practices Incompatible embeddings The FAISS index is typically cre- ated using a specific embeddings model. If a different embed- dings model is used during querying (e.g., a different version of sentence-transformers), it may lead to retrieval mismatche

Ask a question (type exit to stop): exit
Thank you!
